In [6]:
# CELL 1 — Load clean data
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

df_blinkit   = pd.read_csv('../data/clean/blinkit_clean.csv')
df_zepto     = pd.read_csv('../data/clean/zepto_clean.csv')
df_bigbasket = pd.read_csv('../data/clean/bigbasket_clean.csv')

print('✅ Clean data loaded')
print(f'Blinkit categories: {df_blinkit["category"].nunique()}')
print(f'Zepto categories:   {df_zepto["category"].nunique()}')
print(f'BigBasket categories: {df_bigbasket["category"].nunique()}')

✅ Clean data loaded
Blinkit categories: 16
Zepto categories:   14
BigBasket categories: 11


In [3]:
# CELL 2 — Build unified category taxonomy
# Map all 3 platforms to 7 master categories

BLINKIT_MAP = {
    'Fruits and Vegetables': 'Fresh Produce',
    'Snack Foods': 'Snacks',
    'Frozen Foods': 'Snacks',
    'Dairy': 'Dairy & Eggs',
    'Dairy Alternates': 'Dairy & Eggs',
    'Breads': 'Staples & Grains',
    'Breakfast': 'Staples & Grains',
    'Canned': 'Staples & Grains',
    'Soft Drinks': 'Beverages',
    'Hard Drinks': 'Beverages',
    'Health and Hygiene': 'Personal Care',
    'Starchy Foods': 'Staples & Grains',
    'Baking Goods': 'Staples & Grains',
    'Meat': 'Fresh Produce',
    'Seafood': 'Fresh Produce',
    'Household': 'Household',
    'Others': 'Others'
}

ZEPTO_MAP = {
    'Fruits & Vegetables': 'Fresh Produce',
    'Snacks & Munchies': 'Snacks',
    'Dairy & Breakfast': 'Dairy & Eggs',
    'Beverages': 'Beverages',
    'Cold Drinks & Juices': 'Beverages',
    'Bakery & Biscuits': 'Staples & Grains',
    'Atta, Rice & Dal': 'Staples & Grains',
    'Breakfast & Instant Food': 'Staples & Grains',
    'Personal Care': 'Personal Care',
    'Home & Office': 'Household',
    'Eggs, Meat & Fish': 'Fresh Produce',
    'Baby Care': 'Personal Care',
    'Pet Care': 'Household',
    'Cleaning Essentials': 'Household',
    'Paan Corner': 'Others',
    'Ice Creams & Desserts': 'Snacks'
}

BIGBASKET_MAP = {
    'Fruits & Vegetables': 'Fresh Produce',
    'Snacks & Branded Foods': 'Snacks',
    'Beverages': 'Beverages',
    'Bakery & Dairy': 'Dairy & Eggs',
    'Eggs, Meat & Fish': 'Fresh Produce',
    'Foodgrains, Oil & Masala': 'Staples & Grains',
    'Gourmet & World Food': 'Staples & Grains',
    'Beauty & Hygiene': 'Personal Care',
    'Cleaning & Household': 'Household',
    'Kitchen, Garden & Pets': 'Household',
    'Baby Care': 'Personal Care'
}

# Apply mappings
df_blinkit['master_category']   = df_blinkit['category'].map(BLINKIT_MAP).fillna('Others')
df_zepto['master_category']     = df_zepto['category'].map(ZEPTO_MAP).fillna('Others')
df_bigbasket['master_category'] = df_bigbasket['category'].map(BIGBASKET_MAP).fillna('Others')

print('✅ Category taxonomy applied')
print('\nBlinkit master categories:')
print(df_blinkit['master_category'].value_counts())

✅ Category taxonomy applied

Blinkit master categories:
master_category
Snacks              2056
Staples & Grains    1806
Fresh Produce       1721
Household            910
Dairy & Eggs         682
Beverages            659
Personal Care        520
Others               169
Name: count, dtype: int64


In [4]:
# CELL 3 — Compute avg sale price per platform per master category
blinkit_avg = df_blinkit.groupby('master_category')['sale_price'].median().reset_index()
blinkit_avg.columns = ['category', 'Blinkit']

zepto_avg = df_zepto.groupby('master_category')['sale_price'].median().reset_index()
zepto_avg.columns = ['category', 'Zepto']

bigbasket_avg = df_bigbasket.groupby('master_category')['sale_price'].median().reset_index()
bigbasket_avg.columns = ['category', 'BigBasket']

# Merge all 3
price_matrix = blinkit_avg.merge(zepto_avg, on='category', how='outer')
price_matrix = price_matrix.merge(bigbasket_avg, on='category', how='outer')

# Remove Others
price_matrix = price_matrix[price_matrix['category'] != 'Others'].reset_index(drop=True)

# Compute market average
price_matrix['market_avg'] = price_matrix[['Blinkit','Zepto','BigBasket']].mean(axis=1)

# Compute gap % vs market average
for platform in ['Blinkit', 'Zepto', 'BigBasket']:
    price_matrix[f'{platform}_gap%'] = ((price_matrix[platform] - price_matrix['market_avg']) / price_matrix['market_avg'] * 100).round(1)

print('✅ Price matrix built')
print(price_matrix[['category','Blinkit','Zepto','BigBasket','market_avg']].to_string())

✅ Price matrix built
           category   Blinkit    Zepto  BigBasket   market_avg
0         Beverages  144.3444   9500.0     175.00  3273.114800
1      Dairy & Eggs  147.5405      NaN        NaN   147.540500
2     Fresh Produce  146.6418   2800.0      64.00  1003.547267
3         Household  153.3182      NaN     229.00   191.159100
4     Personal Care  128.0349  16200.0     249.00  5525.678300
5            Snacks  143.7628      NaN      87.12   115.441400
6  Staples & Grains  127.0494      NaN     180.50   153.774700


In [9]:
# CELL 4 — Price Heatmap (fixed for VS Code)
import os
import plotly.graph_objects as go

os.makedirs('../outputs', exist_ok=True)

gap_data = price_matrix[['category', 'Blinkit_gap%', 'Zepto_gap%', 'BigBasket_gap%']].copy()
gap_data = gap_data.set_index('category')
gap_data.columns = ['Blinkit', 'Zepto', 'BigBasket']

fig = go.Figure(data=go.Heatmap(
    z=gap_data.values,
    x=gap_data.columns.tolist(),
    y=gap_data.index.tolist(),
    colorscale='RdYlGn_r',
    zmid=0,
    text=[[f'{v:+.1f}%' for v in row] for row in gap_data.values],
    texttemplate='%{text}',
    textfont={"size": 13},
    colorbar=dict(title='Price Gap %', ticksuffix='%')
))

fig.update_layout(
    title='🛒 QC Price Intelligence Matrix',
    height=500,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=13),
    xaxis=dict(side='top', color='white'),
    yaxis=dict(color='white')
)

# Save as HTML only — open in browser to view
fig.write_html('../outputs/price_intelligence_heatmap.html')
print('✅ Heatmap saved!')
print('📂 Go to outputs folder → open price_intelligence_heatmap.html in Chrome')

✅ Heatmap saved!
📂 Go to outputs folder → open price_intelligence_heatmap.html in Chrome


In [16]:
# CELL 5 — Key Findings (write these — they go in your README and LinkedIn)
print('='*55)
print('🔍 KEY FINDINGS — Price Intelligence')
print('='*55)

for _, row in price_matrix.iterrows():
    cheapest = min(['Blinkit','Zepto','BigBasket'], key=lambda p: row[p] if pd.notna(row[p]) else 99999)
    expensive = max(['Blinkit','Zepto','BigBasket'], key=lambda p: row[p] if pd.notna(row[p]) else 0)
    print(f'\n📦 {row["category"]}')
    print(f'   Cheapest:   {cheapest} (₹{row[cheapest]:.0f} median)')
    print(f'   Expensive:  {expensive} (₹{row[expensive]:.0f} median)')
    print(f'   Price gap:  {abs(row[expensive] - row[cheapest]):.0f} difference')

🔍 KEY FINDINGS — Price Intelligence

📦 Beverages
   Cheapest:   Blinkit (₹144 median)
   Expensive:  Zepto (₹9500 median)
   Price gap:  9356 difference

📦 Dairy & Eggs
   Cheapest:   Blinkit (₹148 median)
   Expensive:  Blinkit (₹148 median)
   Price gap:  0 difference

📦 Fresh Produce
   Cheapest:   BigBasket (₹64 median)
   Expensive:  Zepto (₹2800 median)
   Price gap:  2736 difference

📦 Household
   Cheapest:   Blinkit (₹153 median)
   Expensive:  BigBasket (₹229 median)
   Price gap:  76 difference

📦 Personal Care
   Cheapest:   Blinkit (₹128 median)
   Expensive:  Zepto (₹16200 median)
   Price gap:  16072 difference

📦 Snacks
   Cheapest:   BigBasket (₹87 median)
   Expensive:  Blinkit (₹144 median)
   Price gap:  57 difference

📦 Staples & Grains
   Cheapest:   Blinkit (₹127 median)
   Expensive:  BigBasket (₹180 median)
   Price gap:  53 difference


In [17]:
# Add master_category to zepto_clean
df_zepto_clean = pd.read_csv('../data/clean/zepto_clean.csv')
df_zepto_clean['master_category'] = df_zepto_clean['category'].map(ZEPTO_MAP).fillna('Others')
# CELL 6 — Discount Analysis (fixed)
import plotly.express as px

# Fix: use df_zepto_clean not df_zepto
discount_zepto = df_zepto_clean.groupby('master_category')['discount_pct'].median().reset_index()
discount_zepto = discount_zepto[discount_zepto['master_category'] != 'Others']

fig2 = px.bar(
    discount_zepto.sort_values('discount_pct', ascending=True),
    x='discount_pct',
    y='master_category',
    orientation='h',
    title='Zepto — Median Discount % by Category',
    color='discount_pct',
    color_continuous_scale='Reds',
    labels={'discount_pct': 'Discount %', 'master_category': 'Category'}
)

fig2.update_layout(
    height=400,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white'),
    title_font=dict(color='white')
)

fig2.write_html('../outputs/zepto_discount_by_category.html')
print('✅ Discount chart saved!')

✅ Discount chart saved!


In [13]:
# CELL 7 — Save updated data + Day 3 complete
import os
os.makedirs('../outputs', exist_ok=True)

price_matrix.to_csv('../data/clean/price_matrix.csv', index=False)

print('🎉 DAY 3 COMPLETE!')
print('='*55)
print('What you built today:')
print('  ✅ Unified category taxonomy (7 master categories)')
print('  ✅ Price matrix (Blinkit vs Zepto vs BigBasket)')
print('  ✅ Competitive heatmap — THE signature visual')
print('  ✅ Discount analysis by category')
print('  ✅ Key findings written in plain English')
print('='*55)
print('Day 4 tomorrow: Basket Association Rules (Apriori) 🛒')

🎉 DAY 3 COMPLETE!
What you built today:
  ✅ Unified category taxonomy (7 master categories)
  ✅ Price matrix (Blinkit vs Zepto vs BigBasket)
  ✅ Competitive heatmap — THE signature visual
  ✅ Discount analysis by category
  ✅ Key findings written in plain English
Day 4 tomorrow: Basket Association Rules (Apriori) 🛒
